In [2]:
import pandas as pd
import numpy as np

# =====================================
# 1. LOAD DATA
# =====================================
def load_data(file_path, time_col=None):
    df = pd.read_csv(file_path)

    df.columns = df.columns.str.strip()

    if time_col and time_col in df.columns:
        try:
            df = df.sort_values(by=time_col)
        except Exception:
            pass

    return df


# =====================================
# 2. EXTRACT MULTI-SENSOR DATA
# =====================================
def get_sensor_data(df, time_col=None):
    if time_col and time_col in df.columns:
        sensor_df = df.drop(columns=[time_col])
    else:
        sensor_df = df.copy()

    sensor_df = sensor_df.select_dtypes(include=[np.number])
    return sensor_df


# =====================================
# 3. CREATE ROLLING WINDOWS
# =====================================
def create_windows(sensor_df, window_size, step_size):
    windows = []
    n = len(sensor_df)

    for start in range(0, n - window_size + 1, step_size):
        end = start + window_size
        window = sensor_df.iloc[start:end].reset_index(drop=True)
        windows.append(window)

    return windows


# =====================================
# 4. STRUCTURE WINDOWS
# =====================================
def structure_windows(windows):
    return np.array([w.values for w in windows])


# =====================================
# 5. CORRELATION STAGE
# =====================================
def compute_correlations(windows):
    results = []

    for i, w in enumerate(windows):
        corr = w.corr()
        results.append({
            "window_id": i,
            "correlation_matrix": corr
        })

    return results


# =====================================
# 6. PIPELINE
# =====================================
def rolling_window_pipeline(file_path,
                            window_size=5,
                            step_size=2,
                            time_col="time"):

    df = load_data(file_path, time_col)
    sensor_df = get_sensor_data(df, time_col)
    windows = create_windows(sensor_df, window_size, step_size)
    structured_data = structure_windows(windows)
    correlations = compute_correlations(windows)

    return windows, structured_data, correlations


# =====================================
# 7. RUN
# =====================================
if __name__ == "__main__":
    file_path = "most_correlated_streams_window.csv"

    windows, structured_data, correlations = rolling_window_pipeline(
        file_path,
        window_size=5,
        step_size=2,
        time_col="time"
    )

    print("Number of windows:", len(windows))
    print("Structured data shape:", structured_data.shape)

    print("\nFirst window (Sensor Data Only):")
    print(windows[0])

    print("\nCorrelation (Window 0):")
    print(correlations[0]["correlation_matrix"])

Number of windows: 502
Structured data shape: (502, 5, 3)

First window (Sensor Data Only):
         s1       s2        s3
0  1.000000  2.00000  0.700000
1  1.010000  1.99995  0.707000
2  1.019999  1.99980  0.713999
3  1.029996  1.99955  0.720997
4  1.039989  1.99920  0.727993

Correlation (Window 0):
          s1        s2        s3
s1  1.000000 -0.958902  1.000000
s2 -0.958902  1.000000 -0.958902
s3  1.000000 -0.958902  1.000000
